# CSCI 2020 — Module 6
## Lecture Notebook — Week 12: argparse + Refactoring into a Mini-Project

Run this notebook top-to-bottom.


In [ ]:
# SETUP (do not edit)
from pathlib import Path

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'output'
SRC_DIR = ROOT / 'src'
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
SRC_DIR.mkdir(exist_ok=True)

print('Working directory:', ROOT)
print('Data dir:', DATA_DIR)
print('Output dir:', OUTPUT_DIR)
print('Src dir:', SRC_DIR)


## Learning goals
- Build a command-line interface (CLI) with `argparse`
- Run scripts with command-line arguments from a notebook
- Refactor a script into CLI / logic / I/O modules
- Think about dependencies (light) and documentation (README)


## The project we will build (small)
A CLI tool that reads `data/input.csv`, cleans it, and writes outputs.

Command examples:
- `python src/cli.py --help`
- `python src/cli.py --input data/input.csv --output output/cleaned.csv --summary output/summary.json`


## Create a small input CSV in `data/`
This ensures the demo runs even if a student folder is missing files.


In [ ]:
import csv
from pathlib import Path

inp = DATA_DIR / 'input.csv'
if not inp.exists():
    rows = [
        ['date','item','category','quantity','unit_price'],
        [' 2026-02-01 ','paper clips','Office','10','$1.25'],
        ['2026-02-02','PENS','Office','5',' $0.99 '],
        ['2026-02-03','hdmi cable','tech','two','$9.99'],
    ]
    with open(inp, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerows(rows)
print('Input file:', inp)


## Step 1: Create `analysis.py` (logic layer)
This file should NOT know anything about CLI parsing.
It should just clean rows and compute summary values.


In [ ]:
%%writefile src/analysis.py
"""Logic layer: cleaning rows and computing summaries."""

from __future__ import annotations
from typing import Dict, List, Any

from utils import normalize_category, safe_int, safe_float_money

def clean_row(row: Dict[str, str]) -> Dict[str, Any]:
    date = (row.get('date', '') or '').strip()
    item = (row.get('item', '') or '').strip().title()
    category = normalize_category(row.get('category', '') or '')

    quantity = safe_int(row.get('quantity', ''), default=0)
    unit_price = safe_float_money(row.get('unit_price', ''), default=0.0)
    total_value = float(quantity) * float(unit_price)

    return {
        'date': date,
        'item': item,
        'category': category,
        'quantity': int(quantity),
        'unit_price': float(unit_price),
        'total_value': float(total_value),
    }

def compute_summary(cleaned_rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    num_rows_total = len(cleaned_rows)
    num_rows_valid = 0
    total_quantity = 0
    total_value = 0.0
    category_counts: Dict[str, int] = {}
    category_total_value: Dict[str, float] = {}

    for r in cleaned_rows:
        qty = int(r['quantity'])
        val = float(r['total_value'])
        cat = str(r['category'])

        if qty > 0 or float(r['unit_price']) > 0.0:
            num_rows_valid += 1

        total_quantity += qty
        total_value += val

        category_counts[cat] = category_counts.get(cat, 0) + 1
        category_total_value[cat] = category_total_value.get(cat, 0.0) + val

    return {
        'num_rows_total': num_rows_total,
        'num_rows_valid': num_rows_valid,
        'total_quantity': total_quantity,
        'total_value': total_value,
        'category_counts': category_counts,
        'category_total_value': category_total_value,
    }


## Step 2: Create `io_utils.py` (I/O layer)
- Read CSV
- Write CSV
- Write JSON


In [ ]:
%%writefile src/io_utils.py
"""I/O layer: reading and writing files."""

from __future__ import annotations
import csv
import json
from typing import Dict, List, Any

def read_csv(path: str) -> List[Dict[str, str]]:
    rows: List[Dict[str, str]] = []
    with open(path, 'r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
    return rows

def write_cleaned_csv(path: str, cleaned_rows: List[Dict[str, Any]]) -> None:
    fieldnames = ['date','item','category','quantity','unit_price','total_value']
    with open(path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(cleaned_rows)

def write_summary_json(path: str, summary: Dict[str, Any]) -> None:
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2)


## Step 3: Create `cli.py` (CLI layer)
This file should:
- parse arguments
- call I/O and analysis functions
- print clear messages


In [ ]:
%%writefile src/cli.py
"""CLI entrypoint for the mini-project."""

from __future__ import annotations
import argparse
import os

from io_utils import read_csv, write_cleaned_csv, write_summary_json
from analysis import clean_row, compute_summary

def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description='Clean a CSV and produce a JSON summary.'
    )
    parser.add_argument('--input', required=True, help='Path to input CSV')
    parser.add_argument('--output', required=True, help='Path to output cleaned CSV')
    parser.add_argument('--summary', required=True, help='Path to output summary JSON')
    parser.add_argument('--verbose', action='store_true', help='Print extra information')
    return parser

def main() -> None:
    parser = build_parser()
    args = parser.parse_args()

    if not os.path.exists(args.input):
        raise SystemExit(f'Input file not found: {args.input}')

    raw_rows = read_csv(args.input)
    cleaned_rows = [clean_row(r) for r in raw_rows]
    summary = compute_summary(cleaned_rows)

    # Ensure output folders exist
    os.makedirs(os.path.dirname(args.output) or '.', exist_ok=True)
    os.makedirs(os.path.dirname(args.summary) or '.', exist_ok=True)

    write_cleaned_csv(args.output, cleaned_rows)
    write_summary_json(args.summary, summary)

    if args.verbose:
        print('Rows read:', len(raw_rows))
        print('Wrote cleaned CSV:', args.output)
        print('Wrote summary JSON:', args.summary)
        print('Summary keys:', list(summary.keys()))
    else:
        print('Done.')

if __name__ == '__main__':
    main()


## Run your CLI from the notebook
This is a *command line* experience without requiring the terminal.


In [ ]:
!python src/cli.py --help

In [ ]:
!python src/cli.py --input data/input.csv --output output/cleaned.csv --summary output/summary.json --verbose

In [ ]:
# Inspect outputs
from pathlib import Path
print((OUTPUT_DIR / 'cleaned.csv').read_text(encoding='utf-8'))
print((OUTPUT_DIR / 'summary.json').read_text(encoding='utf-8'))


## Refactoring checklist
- CLI should be thin: parse args + call functions
- Logic should be in `analysis.py`
- Reading/writing should be in `io_utils.py`
- Utility helpers in `utils.py`


## Dependencies and environments (awareness)
- In Binder, dependencies are managed by `binder/environment.yml`.
- In local projects, you might use conda envs or requirements files.
- For this course: focus on recognizing dependency issues and documenting how to run.


## Save your work
Before leaving Binder, download your `src/` folder (and outputs if needed) for submission.
